# Inventory Optimization: Reorder Point (ROP) & Safety Stock

In this notebook, we convert demand forecasts into concrete
inventory decisions.

Using forecast errors, demand variability, supplier lead times,
and service level targets, we calculate:
- Safety stock
- Reorder points (ROP)

The objective is to reduce stockouts while avoiding excess inventory.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm

pd.set_option("display.max_columns", None)

import sys
sys.path.append('../')

DATA_PATH_PROCESSED = "../data/processed"

In [2]:
df = pd.read_csv(f"{DATA_PATH_PROCESSED}/feature_engineered_with_segments.csv")
print("Segmented data loaded successfully")

forecast_errors = pd.read_csv(f"{DATA_PATH_PROCESSED}/forecast_error_summary.csv")
print("Forecast error summary loaded successfully")

Segmented data loaded successfully
Forecast error summary loaded successfully


## Service Level Target

NovaCart targets a service level of **96%**, meaning that inventory
should be sufficient to meet demand in 96 out of 100 replenishment cycles.

This service level is converted into a Z-score.

In [4]:
service_level = 0.96
z_score = norm.ppf(service_level)

print(z_score)

1.7506860712521692


## Weekly Demand Statistics per SKU

We compute average weekly demand and demand variability for each SKU.

In [5]:
weekly_stats = (
    df.groupby("sku_id")["units_sold"]
    .agg(
        avg_weekly_demand="mean",
        std_weekly_demand="std"
    )
    .reset_index()
)

## Supplier Lead Time Statistics

Lead time uncertainty increases inventory risk and must be
explicitly factored into safety stock.

In [6]:
lead_time_stats = (
    df.groupby("sku_id")["lead_time_days"]
    .agg(
        avg_lead_time="mean",
        std_lead_time="std"
    )
    .reset_index()
)

lead_time_stats["std_lead_time"] = lead_time_stats["std_lead_time"].fillna(0)

## Combine Demand and Lead Time Statistics

In [7]:
inventory_base = (
    weekly_stats
    .merge(lead_time_stats, on="sku_id", how="left")
)

## Safety Stock Calculation

Safety stock accounts for:
- Demand variability
- Lead time variability
- Desired service level

In [8]:
inventory_base["safety_stock"] = (
    z_score
    * np.sqrt(
        (inventory_base["std_weekly_demand"] ** 2 * inventory_base["avg_lead_time"])
        + (inventory_base["avg_weekly_demand"] ** 2 * inventory_base["std_lead_time"] ** 2)
    )
)

## Reorder Point (ROP) Calculation

ROP defines the inventory level at which a replenishment order
should be placed.

In [9]:
inventory_base["reorder_point"] = (
    inventory_base["avg_weekly_demand"] * inventory_base["avg_lead_time"]
    + inventory_base["safety_stock"]
)

## Attach SKU Segments

SKU segments help adjust inventory policies by importance and risk.

In [10]:
sku_segments = (
    df[["sku_id", "SKU_segment"]]
    .drop_duplicates()
)

inventory_policy = inventory_base.merge(
    sku_segments,
    on="sku_id",
    how="left"
)

## Segment-Level Policy Adjustments

High-value or volatile SKUs are assigned more conservative
inventory buffers.

In [11]:
def segment_multiplier(segment):
    if segment.startswith("A"):
        return 1.2
    elif segment.startswith("B"):
        return 1.0
    else:
        return 0.8

inventory_policy["segment_multiplier"] = (
    inventory_policy["SKU_segment"].apply(segment_multiplier)
)

inventory_policy["adjusted_safety_stock"] = (
    inventory_policy["safety_stock"] * inventory_policy["segment_multiplier"]
)

inventory_policy["adjusted_reorder_point"] = (
    inventory_policy["avg_weekly_demand"] * inventory_policy["avg_lead_time"]
    + inventory_policy["adjusted_safety_stock"]
)

## Final Inventory Policy Snapshot

In [12]:
inventory_policy[
    [
        "sku_id",
        "SKU_segment",
        "avg_weekly_demand",
        "avg_lead_time",
        "adjusted_safety_stock",
        "adjusted_reorder_point"
    ]
].head()

,sku_id,SKU_segment,avg_weekly_demand,avg_lead_time,adjusted_safety_stock,adjusted_reorder_point
0,SKU0001,AX,30.737569,10.0,45.444083,352.819774
1,SKU0002,BX,31.842541,6.0,34.908346,225.963594
2,SKU0003,CX,25.698895,5.0,23.227335,151.721810
3,SKU0004,BX,28.801105,5.0,28.230228,172.235752
4,SKU0005,AX,34.751381,5.0,33.504267,207.261173


## Save Inventory Optimization Output

This table can directly feed into NovaCart’s replenishment workflow.

In [13]:
inventory_policy.to_csv(f"{DATA_PATH_PROCESSED}/inventory_replenishment_policy.csv", index=False)
print("Inventory replenishment policy saved successfully")

Inventory replenishment policy saved successfully


## Key Business Takeaways

- Safety stock is driven by uncertainty, not just average demand
- Lead time variability significantly increases inventory risk
- High-value SKUs require more conservative buffers
- One-size-fits-all inventory rules are inefficient

This inventory policy directly supports:
- Higher service levels
- Fewer stockouts
- Lower excess inventory